# Setup

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib fairlearn shap xgboost kagglehub
import sys
from src import config, visualization, data_loader, feature_engineering, modeling, fairness, visualization, explainability
from src.config import *
from IPython.display import display
sys.path.insert(0, ".")   # makes `src` importable; use "/content" on Colab

You should consider upgrading via the '/Users/Mutru001/Library/Mobile Documents/com~apple~CloudDocs/Personal/UU (cloud)/Period 4/Human Centered ML/project/venv/bin/python3 -m pip install --upgrade pip' command.


In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

# Load data

In [ ]:
DATA_DIR = data_loader.resolve_data_dir()
OUTPUT_DIR = Path("/content/oulad_outputs") if Path("/content").exists() else Path("oulad_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tables = data_loader.load_all_tables(DATA_DIR)
data_loader.inspect_tables(tables["student_info"], tables["courses"])

KaggleHub download failed. Falling back to manual /content/oulad directory.
Error: HTTPSConnectionPool(host='www.kaggle.com', port=443): Read timed out. (read timeout=5)


OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
base = feature_engineering.build_base_table(tables["student_info"], tables["courses"])
early_df = feature_engineering.build_dataset_for_stage(
    base, tables["student_vle"], tables["vle"], tables["courses"],
    tables["student_assess"], tables["assessments"], tables["student_reg"],
    fraction=0.25, stage_name="early_25pct"
)
full_df = feature_engineering.build_dataset_for_stage(
    base, tables["student_vle"], tables["vle"], tables["courses"],
    tables["student_assess"], tables["assessments"], tables["student_reg"],
    fraction=1.00, stage_name="full_100pct"
)
display(early_df.head()); display(full_df.head())


Base table shape: (32593, 14)
Target distribution:
target_unsuccessful
1    17208
0    15385
Name: count, dtype: int64
target_unsuccessful
1    0.527966
0    0.472034
Name: proportion, dtype: float64

Building dataset for early_25pct, fraction=0.25
VLE features: (28360, 18)
Assessment features: (25444, 11)
Registration features: (32593, 6)
Final stage table: (32593, 39)

Building dataset for full_100pct, fraction=1.0
VLE features: (28500, 18)
Assessment features: (25843, 11)
Registration features: (32593, 6)
Final stage table: (32593, 39)


,code_module,code_presentation,id_student,target_unsuccessful,module_presentation_length,gender,age_band,highest_education,imd_band,region,num_of_prev_attempts,studied_credits,date_registration,registered_before_start,days_registered_before_start,vle_total_clicks,vle_active_days,vle_mean_clicks_per_record,vle_max_clicks_per_record,vle_num_records,vle_clicks_per_active_day,clicks_forumng,clicks_homepage,clicks_other_activity,clicks_oucontent,clicks_ouwiki,clicks_quiz,clicks_resource,clicks_subpage,clicks_url,assess_num_submitted,assess_mean_score,assess_min_score,assess_max_score,assess_weighted_score_sum,assess_total_weight_seen,assess_late_rate,assess_mean_days_before_due,stage
0,AAA,2013J,11391,0,268,M,55<=,HE Qualification,90-100%,East Anglian Region,0,240,-159.0,1.0,159.0,447.0,18.0,5.518519,76.0,81.0,24.833333,96.0,63.0,0.0,266.0,0.0,0.0,9.0,12.0,1.0,2.0,81.5,78.0,85.0,24.8,30.0,0.0,1.0,early_25pct
1,AAA,2013J,28400,0,268,F,35-55,HE Qualification,20-30%,Scotland,0,60,-53.0,1.0,53.0,508.0,22.0,3.479452,19.0,146.0,23.090909,181.0,114.0,0.0,148.0,0.0,0.0,0.0,39.0,26.0,2.0,69.0,68.0,70.0,20.6,30.0,0.5,-0.5,early_25pct
2,AAA,2013J,30268,1,268,F,35-55,A Level or Equivalent,30-40%,North Western Region,0,60,-92.0,1.0,92.0,179.0,6.0,4.261905,23.0,42.0,29.833333,87.0,27.0,0.0,52.0,0.0,0.0,0.0,10.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,early_25pct
3,AAA,2013J,31604,0,268,F,35-55,A Level or Equivalent,50-60%,South East Region,0,60,-52.0,1.0,52.0,710.0,38.0,3.212670,13.0,221.0,18.684211,205.0,159.0,0.0,250.0,0.0,0.0,1.0,59.0,36.0,2.0,71.5,71.0,72.0,21.4,30.0,0.0,2.5,early_25pct
4,AAA,2013J,32885,0,268,F,0-35,Lower Than A Level,50-60%,West Midlands Region,0,60,-176.0,1.0,176.0,313.0,21.0,3.226804,19.0,97.0,14.904762,65.0,59.0,0.0,174.0,0.0,0.0,0.0,12.0,3.0,1.0,69.0,69.0,69.0,6.9,10.0,1.0,-7.0,early_25pct


,code_module,code_presentation,id_student,target_unsuccessful,module_presentation_length,gender,age_band,highest_education,imd_band,region,num_of_prev_attempts,studied_credits,date_registration,registered_before_start,days_registered_before_start,vle_total_clicks,vle_active_days,vle_mean_clicks_per_record,vle_max_clicks_per_record,vle_num_records,vle_clicks_per_active_day,clicks_forumng,clicks_homepage,clicks_other_activity,clicks_oucontent,clicks_ouwiki,clicks_quiz,clicks_resource,clicks_subpage,clicks_url,assess_num_submitted,assess_mean_score,assess_min_score,assess_max_score,assess_weighted_score_sum,assess_total_weight_seen,assess_late_rate,assess_mean_days_before_due,stage
0,AAA,2013J,11391,0,268,M,55<=,HE Qualification,90-100%,East Anglian Region,0,240,-159.0,1.0,159.0,836.0,39.0,4.518919,76.0,185.0,21.435897,191.0,131.0,0.0,475.0,0.0,0.0,13.0,21.0,5.0,5.0,82.0,78.0,85.0,82.4,100.0,0.0,1.8,full_100pct
1,AAA,2013J,28400,0,268,F,35-55,HE Qualification,20-30%,Scotland,0,60,-53.0,1.0,53.0,1220.0,73.0,3.253333,23.0,375.0,16.712329,344.0,278.0,10.0,476.0,0.0,0.0,7.0,64.0,41.0,5.0,66.4,60.0,70.0,65.4,100.0,0.4,0.0,full_100pct
2,AAA,2013J,30268,1,268,F,35-55,A Level or Equivalent,30-40%,North Western Region,0,60,-92.0,1.0,92.0,179.0,6.0,4.261905,23.0,42.0,29.833333,87.0,27.0,0.0,52.0,0.0,0.0,0.0,10.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,full_100pct
3,AAA,2013J,31604,0,268,F,35-55,A Level or Equivalent,50-60%,South East Region,0,60,-52.0,1.0,52.0,1989.0,118.0,3.182400,22.0,625.0,16.855932,605.0,402.0,2.0,758.0,0.0,0.0,10.0,125.0,87.0,5.0,76.0,71.0,88.0,76.3,100.0,0.0,2.0,full_100pct
4,AAA,2013J,32885,0,268,F,0-35,Lower Than A Level,50-60%,West Midlands Region,0,60,-176.0,1.0,176.0,739.0,62.0,2.583916,19.0,286.0,11.919355,121.0,152.0,2.0,353.0,0.0,0.0,38.0,62.0,11.0,5.0,54.4,30.0,75.0,55.0,100.0,1.0,-11.4,full_100pct


In [ ]:
train_students, test_students = modeling.student_train_test_split(base)
early_train, early_test = modeling.apply_split(early_df, train_students, test_students)
full_train,  full_test  = modeling.apply_split(full_df,  train_students, test_students)
perf_early, models_early = modeling.fit_and_evaluate_stage("early_25pct", early_train, early_test)
perf_full,  models_full  = modeling.fit_and_evaluate_stage("full_100pct", full_train, full_test)
performance_table = pd.concat([perf_early, perf_full], ignore_index=True)[[
    "stage", "model", "accuracy", "balanced_accuracy",
    "precision_at_risk", "recall_at_risk", "f1_at_risk", "roc_auc"
]]
display(performance_table.sort_values(["model", "stage"]))

Training LogisticRegression for early_25pct...
Training RandomForest for early_25pct...
Training XGBoost for early_25pct...
Training LogisticRegression for full_100pct...
Training RandomForest for full_100pct...
Training XGBoost for full_100pct...


,stage,model,accuracy,balanced_accuracy,precision_at_risk,recall_at_risk,f1_at_risk,roc_auc
0,early_25pct,LogisticRegression,0.786486,0.789048,0.835087,0.740784,0.785114,0.874960
3,full_100pct,LogisticRegression,0.923956,0.924833,0.945132,0.908306,0.926353,0.974703
1,early_25pct,RandomForest,0.811057,0.813844,0.863684,0.761316,0.809276,0.893214
4,full_100pct,RandomForest,0.943243,0.944861,0.976333,0.914372,0.944337,0.983870
2,early_25pct,XGBoost,0.812162,0.814253,0.854826,0.774848,0.812875,0.895021
5,full_100pct,XGBoost,0.948403,0.949539,0.972616,0.928138,0.949857,0.986326


# Analysis

## Fairness

In [ ]:
subgroup_early = fairness.run_fairness_analysis("early_25pct", early_test, models_early)
subgroup_full  = fairness.run_fairness_analysis("full_100pct", full_test,  models_full)
subgroup_metrics = pd.concat([subgroup_early, subgroup_full], ignore_index=True)
fairness_gaps = fairness.fairness_gap_summary(subgroup_metrics)
display(subgroup_metrics.head(30))
display(fairness_gaps.sort_values(["model", "sensitive_attr", "stage"]))

visualization.plot_overall_performance(performance_table, OUTPUT_DIR)
visualization.plot_fairness_gaps(fairness_gaps, OUTPUT_DIR)

## Explainability

In [ ]:
from src.explainability import (
    run_shap_for_tree_model,
    compute_family_importance,
    plot_family_importance,
    subgroup_shap_comparison,
    run_permutation_importance,
    run_partial_dependence,
    plot_shap_distributions
)

plot_shap_distributions(models_early, early_test, sample_size=800, output_dir=OUTPUT_DIR)

RUN_SHAP = True
shap_results = {}
if RUN_SHAP:
    for model_name, pipe in models_early.items():
        shap_results[(model_name, "early_25pct")] = run_shap_for_tree_model(
            pipe,
            early_test,
            "early_25pct",
            model_name,
            output_dir=OUTPUT_DIR,
            shap_available=SHAP_AVAILABLE,
        )
    for model_name, pipe in models_full.items():
        shap_results[(model_name, "full_100pct")] = run_shap_for_tree_model(
            pipe,
            full_test,
            "full_100pct",
            model_name,
            output_dir=OUTPUT_DIR,
            shap_available=SHAP_AVAILABLE,
        )

family_importance = compute_family_importance(shap_results)
plot_family_importance(family_importance, output_dir=OUTPUT_DIR)

if "RandomForest" in models_early and "RandomForest" in models_full:
    if SHAP_AVAILABLE:
        subgroup_shap_comparison(
            models_early["RandomForest"],
            early_test,
            "highest_education",
            "early_25pct",
            "RandomForest",
            shap_available=SHAP_AVAILABLE,
            output_dir=OUTPUT_DIR,
        )
        subgroup_shap_comparison(
            models_early["RandomForest"],
            early_test,
            "imd_band",
            "early_25pct",
            "RandomForest",
            shap_available=SHAP_AVAILABLE,
            output_dir=OUTPUT_DIR,
        )
    else:
        print("Skipping subgroup SHAP comparison because SHAP is unavailable.")

    pfi_early = run_permutation_importance(
        models_early["RandomForest"],
        early_test,
        "early_25pct",
        "RandomForest",
        output_dir=OUTPUT_DIR,
    )
    pfi_full = run_permutation_importance(
        models_full["RandomForest"],
        full_test,
        "full_100pct",
        "RandomForest",
        output_dir=OUTPUT_DIR,
    )

    pdp_features = ["vle_total_clicks", "vle_active_days", "studied_credits"]
    valid_features_early = [f for f in pdp_features if f in early_test.columns]
    valid_features_full = [f for f in pdp_features if f in full_test.columns]

    if valid_features_early:
        run_partial_dependence(
            models_early["RandomForest"],
            early_test,
            "early_25pct",
            "RandomForest",
            valid_features_early,
            output_dir=OUTPUT_DIR,
        )
        run_partial_dependence(
            models_early["RandomForest"],
            early_test,
            "early_25pct",
            "RandomForest",
            valid_features_early,
            centered=True,
            output_dir=OUTPUT_DIR,
        )
    if valid_features_full:
        run_partial_dependence(
            models_full["RandomForest"],
            full_test,
            "full_100pct",
            "RandomForest",
            valid_features_full,
            output_dir=OUTPUT_DIR,
        )
        run_partial_dependence(
            models_full["RandomForest"],
            full_test,
            "full_100pct",
            "RandomForest",
            valid_features_full,
            centered=True,
            output_dir=OUTPUT_DIR,
        )
else:
    print("RandomForest is not available in both early and full models for extra explainability.")


ValueError: Must pass 2-d input. shape=(800, 62, 2)

In [ ]:
from src.reporting import export_tables
export_tables(performance_table, subgroup_metrics, fairness_gaps,
              family_importance=family_importance, output_dir=OUTPUT_DIR)


Saved output files to: oulad_outputs
 - oulad_outputs/fnr_gap_XGBoost.png
 - oulad_outputs/shap_subgroup_RandomForest_early_25pct_imd_band.png
 - oulad_outputs/selection_rate_gap_LogisticRegression.png
 - oulad_outputs/family_importance_RandomForest_full_100pct.png
 - oulad_outputs/fnr_gap_RandomForest.png
 - oulad_outputs/overall_balanced_accuracy_by_stage.png
 - oulad_outputs/family_importance_RandomForest_early_25pct.png
 - oulad_outputs/ice_RandomForest_full_100pct_centered.png
 - oulad_outputs/family_importance_XGBoost_full_100pct.png
 - oulad_outputs/family_importance_XGBoost_early_25pct.png
 - oulad_outputs/feature_family_importance.csv
 - oulad_outputs/ice_RandomForest_early_25pct_centered.png
 - oulad_outputs/shap_subgroup_RandomForest_early_25pct_highest_education.png
 - oulad_outputs/shap_XGBoost_early_25pct.png
 - oulad_outputs/shap_XGBoost_full_100pct.png
 - oulad_outputs/fnr_gap_LogisticRegression.png
 - oulad_outputs/pfi_RandomForest_full_100pct.png
 - oulad_outputs/pfi